# 05D — Nested Tuning and Calibration of Finalist Hold-Probability Models

**Build ID:** `05D_V1_NESTED_TUNING_CALIBRATION_FINALISTS`

05D promotes two primary probability-estimation finalists:

- `E_FULL_EXTENDED` + ordinary row weighting
- `E_FULL_EXTENDED` + incident-equal weighting

For every outer incident-held-out fold, hyperparameter selection and logistic
recalibration are learned only from the already-frozen inner incident folds.

No outer-test tuning and no classification-threshold selection occur here.

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "05D_V1_NESTED_TUNING_CALIBRATION_FINALISTS"

from pathlib import Path
import sys
import subprocess
import importlib.util
import gc
import json
import time

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_table_assembly_v1" / "fireline_hold_modeling_master_v1.parquet"
UNIT_DESIGN_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_design_v1" / "unit_outer_fold_and_training_weights_v1.parquet"
FEATURE_SETS_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "FEATURE_SETS_V1.json"
INNER_FOLDS_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "INNER_INCIDENT_FOLDS_LONG_V1.csv"
BASELINE_05C1_PATH = PROJECT_ROOT / "outputs" / "baseline_calibration_audit_v1" / "ROBUST_OOF_METRICS_AND_CALIBRATION_V1.csv"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "nested_tuning_calibration_v1"
PREDICTION_ROOT = PROJECT_ROOT / "data" / "processed" / "nested_tuning_calibration_v1" / "oof_predictions"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)

for p in [MASTER_PATH, UNIT_DESIGN_PATH, FEATURE_SETS_PATH, INNER_FOLDS_PATH, BASELINE_05C1_PATH]:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
OUTER_FOLDS = 5
INNER_FOLDS = 4
FEATURE_SET_NAME = "E_FULL_EXTENDED"
FINALIST_WEIGHT_SCHEMES = ["row", "incident_equal"]

WEIGHT_COLUMN = {
    "row": "weight_row_raw",
    "incident_equal": "weight_incident_equal_raw",
}

MODEL_SEED = 20260825
MAX_ESTIMATORS = 1000
EARLY_STOPPING_ROUNDS = 50
PRED_EPS = 1e-6

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "lightgbm": "lightgbm",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.special import expit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
import lightgbm as lgb
from lightgbm import LGBMClassifier

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 420)

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)

## Load frozen design and full-extended feature matrix

The 803 predictors are converted to one contiguous `float32` NumPy matrix in a
single operation, avoiding the DataFrame-fragmentation warnings from 05C.

In [ ]:
with FEATURE_SETS_PATH.open("r", encoding="utf-8") as f:
    FEATURE_SETS = json.load(f)

features = FEATURE_SETS[FEATURE_SET_NAME]
assert len(features) == 803

unit_design = pd.read_parquet(UNIT_DESIGN_PATH)
inner_map = pd.read_csv(INNER_FOLDS_PATH)

assert len(unit_design) == EXPECTED_UNITS
assert unit_design["unit_uid"].is_unique
assert unit_design["_irwin_norm"].nunique() == EXPECTED_INCIDENTS

feature_table = pd.read_parquet(MASTER_PATH, columns=["unit_uid", *features])
assert len(feature_table) == EXPECTED_UNITS
assert feature_table["unit_uid"].is_unique

data = unit_design[
    [
        "unit_uid",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "hold_label",
        "outer_fold",
        "weight_row_raw",
        "weight_incident_equal_raw",
    ]
].merge(feature_table, on="unit_uid", how="left", validate="one_to_one")

del feature_table
gc.collect()

X = data[features].to_numpy(dtype=np.float32, na_value=np.nan)
y = data["hold_label"].to_numpy(dtype=np.int8)

assert X.shape == (EXPECTED_UNITS, len(features))
assert np.isin(y, [0, 1]).all()

print("Feature matrix:", X.shape, X.dtype)
print("Approx GiB:", X.nbytes / 1024**3)

BASE_PARAMS = {
    "objective": "binary",
    "learning_rate": 0.05,
    "n_estimators": MAX_ESTIMATORS,
    "max_depth": -1,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "max_bin": 127,
    "n_jobs": -1,
    "verbosity": -1,
    "deterministic": True,
    "force_col_wise": True,
}

TUNING_GRID = [
    {"candidate_id": "C1_LEAVES15_MINCHILD300", "num_leaves": 15, "min_child_samples": 300, "reg_lambda": 2.0},
    {"candidate_id": "C2_LEAVES31_MINCHILD300", "num_leaves": 31, "min_child_samples": 300, "reg_lambda": 2.0},
    {"candidate_id": "C3_LEAVES31_MINCHILD100", "num_leaves": 31, "min_child_samples": 100, "reg_lambda": 1.0},
    {"candidate_id": "C4_LEAVES63_MINCHILD100", "num_leaves": 63, "min_child_samples": 100, "reg_lambda": 1.0},
]

grid_df = pd.DataFrame(TUNING_GRID)
display(grid_df)
grid_df.to_csv(OUTPUT_ROOT / "TUNING_GRID_V1.csv", index=False)

## Helpers

Hyperparameters are chosen by lowest **unweighted inner-OOF Brier score**,
then incident-macro Brier, then log loss. Training weights affect model fitting
only; inner validation metrics remain unweighted.

In [ ]:
def clip_prob(p):
    return np.clip(np.asarray(p, dtype=float), PRED_EPS, 1.0 - PRED_EPS)


def binary_log_loss_vector(y_true, p):
    y_true = np.asarray(y_true, dtype=float)
    p = clip_prob(p)
    return -(y_true * np.log(p) + (1.0 - y_true) * np.log(1.0 - p))


def per_incident_metrics(frame):
    rows = []
    for incident, g in frame.groupby("_irwin_norm", sort=False):
        yt = g["hold_label"].to_numpy(int)
        pp = g["pred_prob"].to_numpy(float)
        mixed = np.unique(yt).size == 2
        rows.append(
            {
                "_irwin_norm": incident,
                "brier": float(np.mean((pp - yt) ** 2)),
                "log_loss": float(np.mean(binary_log_loss_vector(yt, pp))),
                "mixed_class": mixed,
                "pr_auc": float(average_precision_score(yt, pp)) if mixed else np.nan,
                "roc_auc": float(roc_auc_score(yt, pp)) if mixed else np.nan,
            }
        )
    return pd.DataFrame(rows)


def summarize_predictions(frame):
    yt = frame["hold_label"].to_numpy(int)
    pp = frame["pred_prob"].to_numpy(float)
    inc = per_incident_metrics(frame)
    mixed = inc["mixed_class"]

    return {
        "units": len(frame),
        "incidents": frame["_irwin_norm"].nunique(),
        "held_rate": float(np.mean(yt)),
        "mean_pred": float(np.mean(pp)),
        "pooled_pr_auc": float(average_precision_score(yt, pp)),
        "pooled_roc_auc": float(roc_auc_score(yt, pp)),
        "pooled_brier": float(brier_score_loss(yt, pp)),
        "pooled_log_loss": float(log_loss(yt, clip_prob(pp), labels=[0, 1])),
        "incident_macro_brier": float(inc["brier"].mean()),
        "incident_macro_log_loss": float(inc["log_loss"].mean()),
        "mixed_incident_macro_pr_auc": float(inc.loc[mixed, "pr_auc"].mean()),
        "mixed_incident_macro_roc_auc": float(inc.loc[mixed, "roc_auc"].mean()),
    }


def normalized_training_weights(row_indices, scheme):
    raw = data.iloc[row_indices][WEIGHT_COLUMN[scheme]].to_numpy(float)
    assert np.isfinite(raw).all()
    assert (raw > 0).all()
    w = raw * (len(raw) / raw.sum())
    assert np.isclose(w.mean(), 1.0, atol=1e-10, rtol=0)
    return w


def make_model(candidate, seed, n_estimators=MAX_ESTIMATORS):
    params = dict(BASE_PARAMS)
    params.update(
        {
            "num_leaves": int(candidate["num_leaves"]),
            "min_child_samples": int(candidate["min_child_samples"]),
            "reg_lambda": float(candidate["reg_lambda"]),
            "n_estimators": int(n_estimators),
            "random_state": int(seed),
        }
    )
    return LGBMClassifier(**params)


def fit_logistic_recalibrator(y_true, raw_prob, max_iter=100, tol=1e-10, ridge=1e-10):
    y_true = np.asarray(y_true, dtype=float)
    p = clip_prob(raw_prob)
    z = np.log(p / (1.0 - p))
    Z = np.column_stack([np.ones_like(z), z])
    beta = np.array([0.0, 1.0], dtype=float)

    for iteration in range(max_iter):
        eta = Z @ beta
        mu = expit(eta)
        w = mu * (1.0 - mu)
        score = Z.T @ (y_true - mu)
        information = Z.T @ (Z * w[:, None]) + ridge * np.eye(2)
        step = np.linalg.solve(information, score)
        beta = beta + step
        if np.max(np.abs(step)) < tol:
            return {
                "intercept": float(beta[0]),
                "slope": float(beta[1]),
                "iterations": int(iteration + 1),
            }

    raise RuntimeError("Nested logistic recalibrator failed to converge.")


def apply_recalibrator(raw_prob, calibrator):
    p = clip_prob(raw_prob)
    z = np.log(p / (1.0 - p))
    return expit(calibrator["intercept"] + calibrator["slope"] * z)

## Frozen nested-split audit

In [ ]:
split_rows = []

for outer_fold in range(OUTER_FOLDS):
    test_incidents = set(data.loc[data["outer_fold"] == outer_fold, "_irwin_norm"])
    train_incidents = set(data.loc[data["outer_fold"] != outer_fold, "_irwin_norm"])

    inner_here = inner_map[inner_map["outer_fold"] == outer_fold]
    mapped = set(inner_here["_irwin_norm"])

    split_rows.append(
        {
            "outer_fold": outer_fold,
            "outer_test_incidents": len(test_incidents),
            "outer_train_incidents": len(train_incidents),
            "inner_mapped_incidents": len(mapped),
            "train_exactly_matches_inner_map": train_incidents == mapped,
            "zero_outer_test_incidents_in_inner": len(test_incidents & mapped) == 0,
            "all_four_inner_folds_present": set(inner_here["inner_fold"]) == set(range(INNER_FOLDS)),
        }
    )

split_audit = pd.DataFrame(split_rows)
display(split_audit)

assert split_audit["train_exactly_matches_inner_map"].all()
assert split_audit["zero_outer_test_incidents_in_inner"].all()
assert split_audit["all_four_inner_folds_present"].all()

split_audit.to_csv(OUTPUT_ROOT / "NESTED_SPLIT_AUDIT_V1.csv", index=False)

## Nested tuning, calibration, and outer prediction

Expected workload: **160 inner fits + 10 final outer fits**.

In [ ]:
tuning_rows = []
fit_rows = []
selection_rows = []
calibration_rows = []
outer_metric_rows = []

oof_frames = {scheme: [] for scheme in FINALIST_WEIGHT_SCHEMES}

for scheme_index, scheme in enumerate(FINALIST_WEIGHT_SCHEMES):
    print()
    print("=" * 120)
    print("WEIGHT SCHEME:", scheme)
    print("=" * 120)

    for outer_fold in range(OUTER_FOLDS):
        print()
        print("-" * 120)
        print("OUTER FOLD:", outer_fold)
        print("-" * 120)

        outer_train_mask = data["outer_fold"].to_numpy() != outer_fold
        outer_test_mask = ~outer_train_mask

        outer_train_idx = np.flatnonzero(outer_train_mask)
        outer_test_idx = np.flatnonzero(outer_test_mask)

        inner_here = inner_map[inner_map["outer_fold"] == outer_fold][
            ["_irwin_norm", "inner_fold"]
        ].copy()

        incident_to_inner = dict(
            zip(
                inner_here["_irwin_norm"],
                inner_here["inner_fold"].astype(int),
            )
        )

        inner_fold_for_row = data["_irwin_norm"].map(incident_to_inner).to_numpy()

        candidate_records = {}
        candidate_oof = {}

        for candidate_index, candidate in enumerate(TUNING_GRID):
            cid = candidate["candidate_id"]
            inner_oof_prob = np.full(len(data), np.nan, dtype=np.float64)
            best_iterations = []
            total_fit_seconds = 0.0

            for inner_fold in range(INNER_FOLDS):
                inner_val_mask = outer_train_mask & (inner_fold_for_row == inner_fold)
                inner_train_mask = outer_train_mask & ~inner_val_mask

                inner_train_idx = np.flatnonzero(inner_train_mask)
                inner_val_idx = np.flatnonzero(inner_val_mask)

                model = make_model(
                    candidate,
                    MODEL_SEED
                    + 10_000 * scheme_index
                    + 1_000 * outer_fold
                    + 100 * candidate_index
                    + inner_fold,
                )

                weights = normalized_training_weights(inner_train_idx, scheme)

                t0 = time.time()

                model.fit(
                    X[inner_train_idx],
                    y[inner_train_idx],
                    sample_weight=weights,
                    eval_set=[(X[inner_val_idx], y[inner_val_idx])],
                    eval_metric="binary_logloss",
                    callbacks=[
                        lgb.early_stopping(
                            EARLY_STOPPING_ROUNDS,
                            verbose=False,
                        )
                    ],
                )

                elapsed = time.time() - t0
                total_fit_seconds += elapsed

                best_iteration = int(
                    model.best_iteration_
                    if model.best_iteration_ is not None
                    else MAX_ESTIMATORS
                )

                best_iterations.append(best_iteration)

                pred = model.predict_proba(
                    X[inner_val_idx],
                    num_iteration=best_iteration,
                )[:, 1]

                inner_oof_prob[inner_val_idx] = clip_prob(pred)

                fit_rows.append(
                    {
                        "weight_scheme": scheme,
                        "outer_fold": outer_fold,
                        "candidate_id": cid,
                        "inner_fold": inner_fold,
                        "train_units": len(inner_train_idx),
                        "validation_units": len(inner_val_idx),
                        "best_iteration": best_iteration,
                        "fit_seconds": elapsed,
                    }
                )

                del model
                del weights
                gc.collect()

            assert np.isfinite(inner_oof_prob[outer_train_idx]).all()

            inner_frame = data.iloc[outer_train_idx][
                ["unit_uid", "_irwin_norm", "hold_label"]
            ].copy()

            inner_frame["pred_prob"] = inner_oof_prob[outer_train_idx]
            metrics = summarize_predictions(inner_frame)

            rec = {
                "weight_scheme": scheme,
                "outer_fold": outer_fold,
                "candidate_id": cid,
                "num_leaves": candidate["num_leaves"],
                "min_child_samples": candidate["min_child_samples"],
                "reg_lambda": candidate["reg_lambda"],
                "median_best_iteration": float(np.median(best_iterations)),
                "min_best_iteration": int(np.min(best_iterations)),
                "max_best_iteration": int(np.max(best_iterations)),
                "total_inner_fit_seconds": total_fit_seconds,
                **{f"inner_{k}": v for k, v in metrics.items()},
            }

            tuning_rows.append(rec)
            candidate_records[cid] = rec
            candidate_oof[cid] = inner_oof_prob

            print(
                cid,
                "| Brier", f"{metrics['pooled_brier']:.5f}",
                "| MacroBrier", f"{metrics['incident_macro_brier']:.5f}",
                "| LogLoss", f"{metrics['pooled_log_loss']:.5f}",
                "| median trees", f"{np.median(best_iterations):.0f}",
            )

        candidate_table = pd.DataFrame(list(candidate_records.values())).sort_values(
            [
                "inner_pooled_brier",
                "inner_incident_macro_brier",
                "inner_pooled_log_loss",
                "candidate_id",
            ]
        ).reset_index(drop=True)

        selected = candidate_table.iloc[0]
        selected_id = selected["candidate_id"]

        selected_candidate = next(
            c for c in TUNING_GRID
            if c["candidate_id"] == selected_id
        )

        final_trees = int(
            np.clip(
                np.rint(selected["median_best_iteration"]),
                25,
                MAX_ESTIMATORS,
            )
        )

        calibrator = fit_logistic_recalibrator(
            y[outer_train_idx],
            candidate_oof[selected_id][outer_train_idx],
        )

        selection_rows.append(
            {
                "weight_scheme": scheme,
                "outer_fold": outer_fold,
                "selected_candidate_id": selected_id,
                "final_n_estimators": final_trees,
                "inner_pooled_brier": selected["inner_pooled_brier"],
                "inner_incident_macro_brier": selected["inner_incident_macro_brier"],
                "inner_pooled_log_loss": selected["inner_pooled_log_loss"],
            }
        )

        calibration_rows.append(
            {
                "weight_scheme": scheme,
                "outer_fold": outer_fold,
                "calibration_intercept": calibrator["intercept"],
                "calibration_slope": calibrator["slope"],
                "calibration_iterations": calibrator["iterations"],
            }
        )

        final_model = make_model(
            selected_candidate,
            MODEL_SEED + 100_000 + 10_000 * scheme_index + outer_fold,
            n_estimators=final_trees,
        )

        final_weights = normalized_training_weights(outer_train_idx, scheme)

        t0 = time.time()

        final_model.fit(
            X[outer_train_idx],
            y[outer_train_idx],
            sample_weight=final_weights,
        )

        final_fit_seconds = time.time() - t0

        raw_prob = clip_prob(
            final_model.predict_proba(
                X[outer_test_idx]
            )[:, 1]
        )

        cal_prob = apply_recalibrator(
            raw_prob,
            calibrator,
        )

        outer_frame = data.iloc[outer_test_idx][
            [
                "unit_uid",
                "_irwin_norm",
                "IncidentName",
                "Year",
                "hold_label",
                "outer_fold",
            ]
        ].copy()

        outer_frame["pred_prob_raw"] = raw_prob
        outer_frame["pred_prob_calibrated"] = cal_prob
        outer_frame["weight_scheme"] = scheme
        outer_frame["selected_candidate_id"] = selected_id
        outer_frame["final_n_estimators"] = final_trees

        oof_frames[scheme].append(outer_frame)

        raw_eval = outer_frame[
            ["unit_uid", "_irwin_norm", "hold_label"]
        ].copy()
        raw_eval["pred_prob"] = raw_prob

        cal_eval = outer_frame[
            ["unit_uid", "_irwin_norm", "hold_label"]
        ].copy()
        cal_eval["pred_prob"] = cal_prob

        raw_metrics = summarize_predictions(raw_eval)
        cal_metrics = summarize_predictions(cal_eval)

        outer_metric_rows.append(
            {
                "weight_scheme": scheme,
                "outer_fold": outer_fold,
                "selected_candidate_id": selected_id,
                "final_n_estimators": final_trees,
                "final_fit_seconds": final_fit_seconds,
                **{f"raw_{k}": v for k, v in raw_metrics.items()},
                **{f"cal_{k}": v for k, v in cal_metrics.items()},
            }
        )

        print("SELECTED:", selected_id, "| trees:", final_trees, "| calibrator:", calibrator)
        print(
            f"RAW PR={raw_metrics['pooled_pr_auc']:.4f} "
            f"ROC={raw_metrics['pooled_roc_auc']:.4f} "
            f"Brier={raw_metrics['pooled_brier']:.4f}"
        )
        print(
            f"CAL PR={cal_metrics['pooled_pr_auc']:.4f} "
            f"ROC={cal_metrics['pooled_roc_auc']:.4f} "
            f"Brier={cal_metrics['pooled_brier']:.4f}"
        )

        del final_model
        del final_weights
        del candidate_oof
        gc.collect()

tuning_results = pd.DataFrame(tuning_rows)
inner_fit_log = pd.DataFrame(fit_rows)
selected_configs = pd.DataFrame(selection_rows)
calibration_params = pd.DataFrame(calibration_rows)
outer_fold_metrics = pd.DataFrame(outer_metric_rows)

tuning_results.to_csv(OUTPUT_ROOT / "INNER_TUNING_RESULTS_V1.csv", index=False)
inner_fit_log.to_csv(OUTPUT_ROOT / "INNER_FIT_LOG_V1.csv", index=False)
selected_configs.to_csv(OUTPUT_ROOT / "SELECTED_CONFIGS_BY_OUTER_FOLD_V1.csv", index=False)
calibration_params.to_csv(OUTPUT_ROOT / "NESTED_CALIBRATION_PARAMS_V1.csv", index=False)
outer_fold_metrics.to_csv(OUTPUT_ROOT / "OUTER_FOLD_RAW_AND_CALIBRATED_METRICS_V1.csv", index=False)

## Assemble nested OOF predictions and compare with 05C1

In [ ]:
prediction_rows = []
summary_rows = []

for scheme in FINALIST_WEIGHT_SCHEMES:
    oof = pd.concat(oof_frames[scheme], ignore_index=True)

    assert len(oof) == EXPECTED_UNITS
    assert oof["unit_uid"].is_unique
    assert oof["_irwin_norm"].nunique() == EXPECTED_INCIDENTS
    assert np.isfinite(
        oof[
            ["pred_prob_raw", "pred_prob_calibrated"]
        ].to_numpy(float)
    ).all()

    path = PREDICTION_ROOT / f"E_FULL_EXTENDED__{scheme}__nested_tuned.parquet"
    oof.to_parquet(path, index=False)

    prediction_rows.append(
        {
            "weight_scheme": scheme,
            "rows": len(oof),
            "unique_units": oof["unit_uid"].nunique(),
            "unique_incidents": oof["_irwin_norm"].nunique(),
            "path": str(path),
        }
    )

    for probability_type, column in [
        ("raw", "pred_prob_raw"),
        ("nested_calibrated", "pred_prob_calibrated"),
    ]:
        eval_frame = oof[
            ["unit_uid", "_irwin_norm", "hold_label"]
        ].copy()
        eval_frame["pred_prob"] = oof[column]

        summary_rows.append(
            {
                "weight_scheme": scheme,
                "probability_type": probability_type,
                **summarize_predictions(eval_frame),
            }
        )

prediction_audit = pd.DataFrame(prediction_rows)
final_oof_summary = pd.DataFrame(summary_rows)

display(prediction_audit)
display(final_oof_summary)

prediction_audit.to_csv(
    OUTPUT_ROOT / "NESTED_OOF_PREDICTION_AUDIT_V1.csv",
    index=False,
)

final_oof_summary.to_csv(
    OUTPUT_ROOT / "FINAL_NESTED_OOF_METRICS_V1.csv",
    index=False,
)

baseline = pd.read_csv(BASELINE_05C1_PATH)

baseline_ids = {
    "row": "E_FULL_EXTENDED__row",
    "incident_equal": "E_FULL_EXTENDED__incident_equal",
}

comparison_rows = []

for scheme in FINALIST_WEIGHT_SCHEMES:
    old = baseline[
        baseline["experiment_id"] == baseline_ids[scheme]
    ].iloc[0]

    raw = final_oof_summary[
        (final_oof_summary["weight_scheme"] == scheme)
        & (final_oof_summary["probability_type"] == "raw")
    ].iloc[0]

    cal = final_oof_summary[
        (final_oof_summary["weight_scheme"] == scheme)
        & (final_oof_summary["probability_type"] == "nested_calibrated")
    ].iloc[0]

    comparison_rows.append(
        {
            "weight_scheme": scheme,
            "baseline_pr_auc": old["pooled_pr_auc"],
            "tuned_raw_pr_auc": raw["pooled_pr_auc"],
            "tuned_calibrated_pr_auc": cal["pooled_pr_auc"],
            "baseline_roc_auc": old["pooled_roc_auc"],
            "tuned_raw_roc_auc": raw["pooled_roc_auc"],
            "baseline_brier": old["pooled_brier"],
            "tuned_raw_brier": raw["pooled_brier"],
            "tuned_calibrated_brier": cal["pooled_brier"],
            "baseline_log_loss": old["pooled_log_loss"],
            "tuned_raw_log_loss": raw["pooled_log_loss"],
            "tuned_calibrated_log_loss": cal["pooled_log_loss"],
            "baseline_incident_macro_brier": old["incident_macro_brier"],
            "tuned_calibrated_incident_macro_brier": cal["incident_macro_brier"],
            "baseline_mixed_macro_roc": old["mixed_incident_macro_roc_auc"],
            "tuned_calibrated_mixed_macro_roc": cal["mixed_incident_macro_roc_auc"],
        }
    )

baseline_comparison = pd.DataFrame(comparison_rows)
display(baseline_comparison)

baseline_comparison.to_csv(
    OUTPUT_ROOT / "TUNED_VS_BASELINE_COMPARISON_V1.csv",
    index=False,
)

selection_frequency = (
    selected_configs.groupby(
        ["weight_scheme", "selected_candidate_id"],
        as_index=False,
    )
    .agg(
        outer_folds_selected=("outer_fold", "size"),
        median_final_trees=("final_n_estimators", "median"),
        min_final_trees=("final_n_estimators", "min"),
        max_final_trees=("final_n_estimators", "max"),
    )
)

display(selected_configs)
display(selection_frequency)
display(calibration_params)

selection_frequency.to_csv(
    OUTPUT_ROOT / "SELECTION_FREQUENCY_V1.csv",
    index=False,
)

## Final checks and report

In [ ]:
checks = pd.DataFrame(
    [
        {
            "check": "exact_two_probability_finalists",
            "pass": set(FINALIST_WEIGHT_SCHEMES) == {"row", "incident_equal"},
        },
        {
            "check": "all_outer_train_sets_match_frozen_inner_maps",
            "pass": bool(split_audit["train_exactly_matches_inner_map"].all()),
        },
        {
            "check": "zero_outer_test_incidents_used_in_inner_tuning",
            "pass": bool(split_audit["zero_outer_test_incidents_in_inner"].all()),
        },
        {
            "check": "exact_40_candidate_outer_combinations",
            "pass": len(tuning_results)
            == len(FINALIST_WEIGHT_SCHEMES) * OUTER_FOLDS * len(TUNING_GRID),
        },
        {
            "check": "exact_160_inner_fits",
            "pass": len(inner_fit_log)
            == len(FINALIST_WEIGHT_SCHEMES)
            * OUTER_FOLDS
            * len(TUNING_GRID)
            * INNER_FOLDS,
        },
        {
            "check": "exact_10_final_outer_models",
            "pass": len(selected_configs)
            == len(FINALIST_WEIGHT_SCHEMES) * OUTER_FOLDS,
        },
        {
            "check": "exact_10_nested_calibrators",
            "pass": len(calibration_params)
            == len(FINALIST_WEIGHT_SCHEMES) * OUTER_FOLDS,
        },
        {
            "check": "both_nested_oof_sets_304143_unique_units",
            "pass": bool(
                (prediction_audit["rows"] == EXPECTED_UNITS).all()
                and (prediction_audit["unique_units"] == EXPECTED_UNITS).all()
            ),
        },
        {
            "check": "both_nested_oof_sets_381_incidents",
            "pass": bool(
                (prediction_audit["unique_incidents"] == EXPECTED_INCIDENTS).all()
            ),
        },
        {
            "check": "no_outer_test_hyperparameter_tuning",
            "pass": True,
        },
        {
            "check": "no_outer_test_calibration_fitting",
            "pass": True,
        },
        {
            "check": "no_threshold_selection",
            "pass": True,
        },
    ]
)

display(checks)

PASS = bool(checks["pass"].all())

print()
print(
    "05D NESTED TUNING / CALIBRATION:",
    "PASS / FINALISTS READY FOR PRIMARY-MODEL SELECTION"
    if PASS
    else "REVIEW REQUIRED",
)

checks.to_csv(
    OUTPUT_ROOT / "FINAL_NESTED_TUNING_CHECKS_V1.csv",
    index=False,
)

def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report_lines = [
    "# 05D Nested Tuning and Calibration of Finalist Models",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    "## Selected configurations",
    "",
    tomd(selected_configs),
    "",
    "## Selection frequency",
    "",
    tomd(selection_frequency),
    "",
    "## Nested calibration parameters",
    "",
    tomd(calibration_params),
    "",
    "## Final nested OOF metrics",
    "",
    tomd(final_oof_summary),
    "",
    "## Tuned versus 05C1 baseline",
    "",
    tomd(baseline_comparison),
    "",
    "## Final checks",
    "",
    tomd(checks),
    "",
    "## Decision",
    "",
    "`PASS / FINALISTS READY FOR PRIMARY-MODEL SELECTION`"
    if PASS
    else "`REVIEW REQUIRED`",
]

REPORT_PATH = OUTPUT_ROOT / "NESTED_TUNING_CALIBRATION_REPORT_V1.md"
REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)

print("Saved:", REPORT_PATH)